In [1]:
import random
import time
import pandas as pd
import plotly.express as px
from haversine import haversine
from fast_module import fast_haversine_batch

In [2]:
def make_points(n, seed=42):
    rng = random.Random(seed)

    lat1, lat2, lon1, lon2 = [], [], [], []

    for _ in range(n):
        lat1.append(rng.uniform(1.25, 1.47))
        lon1.append(rng.uniform(103.60, 104.05))
        lat2.append(rng.uniform(1.25, 1.47))
        lon2.append(rng.uniform(103.60, 104.05))

    return lat1, lat2, lon1, lon2

In [3]:
def python_haversine_batch(lat1, lon1, lat2, lon2):
    return [haversine((a, b), (c, d)) for a, b, c, d in zip(lat1, lat2, lon1, lon2)]

In [4]:
results = []

In [5]:
for n in [1_000, 10_000, 100_000, 1_000_000]:
    lat1, lon1, lat2, lon2 = make_points(n)


    start = time.perf_counter()
    py_result = python_haversine_batch(lat1, lon1, lat2, lon2)
    py_time = time.perf_counter() - start

    start = time.perf_counter()
    cpp_result = fast_haversine_batch(lat1, lon1, lat2, lon2)
    cpp_time = time.perf_counter() - start

    max_error = max(abs(a - b) for a, b in zip(py_result, cpp_result))

    result_dict = {
        "n" : n,
        "python_time" : py_time,
        "cpp_omp_time" : cpp_time,
        "speedup" : py_time / cpp_time,
        "max_error" : max_error
        }
        

    results.append(result_dict)

In [6]:
bench = pd.DataFrame(results)
bench

,n,python_time,cpp_omp_time,speedup,max_error
0,1000,0.000692,0.000831,0.831850,2.827960e-12
1,10000,0.010937,0.000571,19.141057,2.835066e-12
2,100000,0.094058,0.004087,23.013433,2.835066e-12
3,1000000,0.670738,0.043373,15.464304,2.835066e-12


In [7]:
runtime_long = bench.melt(id_vars="n", value_vars=["python_time", "cpp_omp_time"], var_name="implementation", value_name="seconds")

In [8]:
runtime_fig = px.line(
    runtime_long,
    x="n",
    y="seconds",
    color="implementation",
    markers=True,
    log_x=True,
    log_y=True,
    title="Haversine Runtime: Python vs. C++ OpenMP"
)

In [9]:
runtime_fig.update_layout(
    xaxis_title= "Number of distance calculations",
    yaxis_title="Runtime (s)"
)

In [21]:
speedup_fig = px.line(
    bench,
    x="n",
    y="speedup",
    markers=True,
    log_x=True,
    title="C++ OpenMP Speedup over Python Haversine"
)

In [22]:
speedup_fig.update_layout(
    xaxis_title = "No. of Distance Calculations",
    yaxis_title = "Speedup Factor"
)

speedup_fig.add_hline(
    y=1,
    line_dash="dash",
    annotation_text="Equal performance"
)

speedup_fig.update_yaxes(range=[0, 30])

In [12]:
error_fig = px.line(
    bench,
    x="n",
    y="max_error",
    markers=True,
    log_x=True,
    log_y=True,
    title="Maximum Abs. Error vs Python Haversine",
)

In [13]:
error_fig.update_layout(
    xaxis_title="Number of distance calculations",
    yaxis_title="Max absolute error (km)",
)



In [25]:
import os
print("CPU Count:\t", os.cpu_count())

CPU Count:	 16
